# 🚦 YOLO Traffic Sign & Signal Detector Training

This notebook automates downloading the **JetRacer SmartCity Dataset** from Kaggle, training a **YOLOv8** object detector using **Ultralytics**, and exporting the trained model to **ONNX** format for deployment on the **NVIDIA Jetson Nano**.


In [ ]:
import os
from google.colab import userdata

# 1. Retrieve Kaggle API Token from Colab Secrets
os.environ['KAGGLE_API_TOKEN'] = userdata.get('kaggle')


In [ ]:
!kaggle --version
!kaggle config view


In [ ]:
!rm -f jetracer-smartcity.zip
!rm -rf dataset/


In [ ]:
# Download JetRacer Smart City dataset from Kaggle
!kaggle datasets download -d daf2pro/jetracer-smartcity --unzip -p ./dataset


In [ ]:
os.listdir("dataset")


In [ ]:
# Install Ultralytics and dependencies
!pip install ultralytics pycocotools


In [ ]:
import yaml
from pathlib import Path

# Verify data.yaml contents
dataset_dir = Path("./dataset")
yaml_path = dataset_dir / "data.yaml"

if yaml_path.exists():
    with open(yaml_path, 'r') as f:
        data_cfg = yaml.safe_load(f)
    print("Dataset Classes:", data_cfg.get('names'))


In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Train YOLOv8 model on Smart City traffic sign dataset
results = model.train(
    data='./dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    workers=4,
    name='jetracer_yolov8n',
    exist_ok=True
)


In [ ]:
# Evaluate model performance on validation set
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)


In [ ]:
# Export trained YOLOv8 model to ONNX format (opset 12 / 15 for TensorRT compatibility)
exported_path = model.export(format='onnx', imgsz=640, opset=12, dynamic=False)
print("Exported ONNX Model Path:", exported_path)
